# Arsitrad Full - Kaggle Technical Notebook

This is the end-to-end Kaggle notebook for Arsitrad v2.

Run every cell in order. This notebook is opinionated on purpose: GGUF setup, answer generation, evaluation dry-run, and Streamlit smoke launch are part of the flow, not side quests.

What it covers:
- clone the shipped `v2.0.0` repo state
- install notebook dependencies plus `llama-cpp-python`
- inspect corpus and config
- bootstrap sparse snapshot automatically if the JSONL artifact is missing
- run dry-run ingest on a subset
- inspect sparse retrieval directly
- run GGUF answer generation
- run the starter evaluation flow
- boot Streamlit and verify it comes up on port 8501

For a faster, lower-drama demo, use `Arsitrad_v2_Kaggle_Demo.ipynb` instead.


## Notebook posture

This notebook now runs the portable Arsitrad path end to end:
- sparse snapshot bootstrap
- retrieval inspection
- GGUF answer generation
- evaluation dry-run
- Streamlit boot check

If you also want full hybrid retrieval, set `ARSITRAD_DATABASE_URL` before the retrieval or answer cells. The notebook will pick it up automatically, but the base sequential run no longer asks you to choose between manual branches.


In [ ]:
%cd /kaggle/working
!rm -rf arsitrad
!git clone --depth 1 --branch v2.0.0 https://github.com/arsitekberotot/arsitrad.git
%cd /kaggle/working/arsitrad
!git rev-parse --short HEAD


In [ ]:
import os
import shutil
import subprocess
import sys

base_packages = [
    'pyyaml',
    'pdfplumber',
    'rank-bm25',
    'requests',
    'pydantic',
    'huggingface_hub',
    'streamlit',
    'pytest',
]

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--upgrade-strategy',
    'only-if-needed',
    *base_packages,
])

has_nvidia = shutil.which('nvidia-smi') is not None and subprocess.run(
    ['nvidia-smi'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

llama_env = os.environ.copy()
llama_cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'llama-cpp-python']
if has_nvidia:
    llama_env['CMAKE_ARGS'] = '-DGGML_CUDA=on'
    llama_env['FORCE_CMAKE'] = '1'
    print('Installing llama-cpp-python with CUDA support for Kaggle GPU...')
else:
    print('No NVIDIA GPU detected; installing CPU llama-cpp-python build.')

subprocess.check_call(llama_cmd, env=llama_env)

import importlib.util
import numpy as np
import pandas as pd

print('Using platform NumPy version:', np.__version__)
print('Using platform pandas version:', pd.__version__)
print('llama_cpp installed:', importlib.util.find_spec('llama_cpp') is not None)
print('Step 2 keeps platform runtime packages intact and installs the GGUF runtime required by this notebook.')


## Runtime assumptions

- This notebook installs `llama-cpp-python` and expects the GGUF download step to succeed later.
- Sparse retrieval works from repo assets alone.
- Full hybrid retrieval still needs your own `ARSITRAD_DATABASE_URL`, but that is an external database prerequisite, not a notebook branch you have to choose manually.


In [ ]:
import json
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import yaml

config = yaml.safe_load((repo_root / 'config.yaml').read_text(encoding='utf-8'))
print(json.dumps(config['v2'], ensure_ascii=False, indent=2)[:4000])


In [ ]:
import importlib
import json
import os
import sys
from collections import Counter
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()

from pipeline.chunker import infer_number, infer_region, infer_year
from pipeline.taxonomy import enrich_metadata, infer_building_use, infer_topic

text_paths = sorted((repo_root / 'data/corpus').rglob('*.txt'))
corpus_path = repo_root / 'data/processed/v2/bm25_corpus.jsonl'

def build_sparse_snapshot_from_tracked_chunks(output_path: Path) -> int:
    chunk_sources = [
        (repo_root / 'data/processed/national_chunks.json', 'national'),
        (repo_root / 'data/processed/local_chunks.json', 'local'),
    ]
    records = []
    for chunk_path, scope in chunk_sources:
        if not chunk_path.exists():
            continue
        payload = json.loads(chunk_path.read_text(encoding='utf-8'))
        chunks = payload.get('chunks', [])
        metadatas = payload.get('metadatas', [])
        ids = payload.get('ids', [])
        for chunk_text, meta, chunk_id in zip(chunks, metadatas, ids):
            source_name = str(meta.get('source') or meta.get('title') or chunk_id)
            is_local = scope == 'local'
            reg_type = str(meta.get('reg_type') or ('Perda' if is_local else 'Unknown')).title()
            year = meta.get('year') or infer_year(source_name)
            number = meta.get('number') or infer_number(source_name)
            region = 'nasional' if not is_local else infer_region(Path(source_name + '.txt'), source_name, is_local=True)
            metadata = {
                'source_name': source_name,
                'source_path': str((Path('/data/corpus/local_regulations') if is_local else Path('/data/corpus/indonesian-construction-law')) / f'{source_name}.txt'),
                'source_file': f'{source_name}.txt',
                'reg_type': reg_type,
                'year': year,
                'number': number,
                'region': region,
                'scope': 'local' if is_local else 'national',
                'island': meta.get('island'),
                'chunk_index': int(meta.get('chunk_idx', 0) or 0),
                'start_page': int(meta.get('page', 1) or 1),
                'end_page': int(meta.get('page', 1) or 1),
                'title': source_name,
            }
            topic = infer_topic(source_name, chunk_text)
            building_use = infer_building_use(source_name, chunk_text)
            if topic:
                metadata['topic'] = topic
                metadata['typology'] = topic
            if building_use:
                metadata['building_use'] = building_use
            metadata = enrich_metadata(metadata, str(chunk_text))
            records.append({
                'chunk_key': str(chunk_id),
                'content': str(chunk_text),
                'metadata': metadata,
                'start_page': metadata['start_page'],
                'end_page': metadata['end_page'],
            })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    return len(records)

needs_bootstrap = True
if corpus_path.exists():
    row_count = sum(1 for line in corpus_path.open('r', encoding='utf-8') if line.strip())
    if row_count > 0:
        needs_bootstrap = False
        print('Using existing sparse corpus snapshot:', corpus_path)
    else:
        print('Sparse snapshot exists but is empty. Rebuilding it from tracked chunk stores...')
if needs_bootstrap:
    print('Sparse snapshot missing. Bootstrapping from tracked chunk stores...')
    built = build_sparse_snapshot_from_tracked_chunks(corpus_path)
    print('Bootstrapped sparse records:', built)

source_counts = Counter()
row_count = 0
with corpus_path.open('r', encoding='utf-8') as handle:
    for line in handle:
        record = json.loads(line)
        row_count += 1
        metadata = record.get('metadata') or {}
        source_counts[metadata.get('source_name', 'UNKNOWN')] += 1

print('Text documents  =', len(text_paths))
print('Sparse rows     =', row_count)
print('Top sources:')
for name, count in source_counts.most_common(10):
    print(f'  - {name}: {count}')


In [ ]:
%cd /kaggle/working/arsitrad
!python -m pipeline.ingest --config config.yaml --dry-run --from-sparse --limit-docs 5


In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()

from pipeline.retriever import HybridRetriever

retriever = HybridRetriever(config_path=str(repo_root / 'config.yaml'))

queries = [
    'Apa bedanya IMB dan PBG?',
    'Apakah RDTR wajib dicek sebelum mengurus PBG?',
    'Apa yang harus dicek untuk bangunan di dekat sungai menurut tata ruang?',
]

for query in queries:
    result = retriever.retrieve(query)
    print('=' * 100)
    print('QUESTION:', query)
    print('STANDALONE:', result.standalone_query)
    print('CONFIDENCE:', f'{result.confidence:.2f}')
    print('SHOULD_ANSWER:', result.should_answer)
    for idx, candidate in enumerate(result.candidates[:3], start=1):
        print(f'  [{idx}] {candidate.metadata.get("source_name", "UNKNOWN")} | region={candidate.metadata.get("region", "-")} | score={candidate.score:.4f}')


## GGUF answer engine

This notebook now treats GGUF generation as a required step.
The next cell downloads the model, and the following cell must answer with `used_model = True`.


In [ ]:
import importlib.util
import os
import sys
from pathlib import Path
from huggingface_hub import hf_hub_download

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

model_dir = repo_root / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'gemma-4-E4B-it-Q4_K_M.gguf'

HAS_LLAMA_CPP = importlib.util.find_spec('llama_cpp') is not None
if not HAS_LLAMA_CPP:
    raise RuntimeError('llama_cpp is missing. Step 2 was supposed to install it for this notebook.')

gguf_path = hf_hub_download(
    repo_id='ggml-org/gemma-4-E4B-it-GGUF',
    filename='gemma-4-E4B-it-Q4_K_M.gguf',
    local_dir=str(model_dir),
)

if not model_path.exists():
    raise FileNotFoundError(model_path)

size_gb = model_path.stat().st_size / (1024 ** 3)
print('GGUF ready at:', gguf_path)
print(f'GGUF size: {size_gb:.2f} GB')
print('llama_cpp installed =', HAS_LLAMA_CPP)
print('model file exists   =', model_path.exists())


In [ ]:
import importlib
import importlib.util
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()

from pipeline.inference import ArsitradAnswerEngine, load_inference_config

class KaggleGGUFInferenceEngine:
    def __init__(self, config, n_gpu_layers: int = -1, n_threads: int = 2):
        self.config = config
        self.n_gpu_layers = n_gpu_layers
        self.n_threads = n_threads
        self._model = None

    @property
    def model(self):
        if self._model is None:
            model_path = Path(self.config.model_path)
            if not model_path.exists():
                raise FileNotFoundError(str(model_path))
            from llama_cpp import Llama
            self._model = Llama(
                model_path=str(model_path),
                n_ctx=self.config.context_window,
                n_gpu_layers=self.n_gpu_layers,
                n_threads=self.n_threads,
                verbose=False,
            )
        return self._model

    def generate(self, prompt: str) -> str:
        result = self.model(
            prompt,
            max_tokens=self.config.max_tokens,
            temperature=self.config.temperature,
            top_p=self.config.top_p,
            repeat_penalty=self.config.repeat_penalty,
            stop=['<end_of_turn>', '<eos>'],
        )
        return result['choices'][0]['text'].strip()

config_path = repo_root / 'config.yaml'
cfg = load_inference_config(config_path)
if importlib.util.find_spec('llama_cpp') is None:
    raise RuntimeError('llama_cpp is still unavailable after the install cell.')
if not Path(cfg.model_path).exists():
    raise FileNotFoundError(cfg.model_path)

engine = ArsitradAnswerEngine(
    config_path=config_path,
    inference_engine=KaggleGGUFInferenceEngine(cfg),
)

result = engine.answer('Apa syarat PBG untuk rumah tinggal 2 lantai di Semarang?')
print('used_model =', result.used_model)
if not result.used_model:
    raise RuntimeError('Expected a GGUF-backed answer, but Arsitrad fell back instead.')
print('confidence =', result.retrieval.confidence)
print(result.answer[:2500])


In [ ]:
%cd /kaggle/working/arsitrad
!python -m pipeline.eval.ragas_eval --questions data/eval/golden_queries.json --output /kaggle/working/arsitrad_eval.json --dry-run

In [ ]:
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
log_path = repo_root / 'streamlit.log'
if log_path.exists():
    log_path.unlink()

env = os.environ.copy()
env['PYTHONPATH'] = str(repo_root) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
cmd = [
    sys.executable,
    '-m',
    'streamlit',
    'run',
    'ui/app.py',
    '--server.headless=true',
    '--server.address=0.0.0.0',
    '--server.port=8501',
]

with log_path.open('w', encoding='utf-8') as log_handle:
    proc = subprocess.Popen(cmd, cwd=repo_root, env=env, stdout=log_handle, stderr=subprocess.STDOUT)

time.sleep(12)
if proc.poll() is not None:
    print(log_path.read_text(encoding='utf-8'))
    raise RuntimeError(f'Streamlit exited early with code {proc.returncode}')

health_error = None
for _ in range(10):
    try:
        urllib.request.urlopen('http://127.0.0.1:8501', timeout=5)
        health_error = None
        break
    except Exception as exc:
        health_error = exc
        time.sleep(2)

if health_error is not None:
    print(log_path.read_text(encoding='utf-8'))
    raise RuntimeError(f'Streamlit started but health check failed: {health_error}')

print('Streamlit is running on http://127.0.0.1:8501')
print('PID:', proc.pid)
print('Log:', log_path)


## When to use which notebook

Use `Arsitrad_v2_Kaggle_Demo.ipynb` when you want:
- lowest setup friction
- cleaner live demo
- safer judge flow

Use `Arsitrad-Full.ipynb` when you want:
- the full sequential technical run
- ingestion / retrieval inspection
- mandatory GGUF generation
- evaluation and Streamlit boot verification
